# Vietnamese  English Translation Fine-Tuning

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configure paths

In [ ]:
!rm -rf /content/translate

In [ ]:
!git clone https://github.com/phuc22062004/translate.git

Cloning into 'translate'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 146 (delta 75), reused 138 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 16.60 MiB | 16.76 MiB/s, done.
Resolving deltas: 100% (75/75), done.


In [ ]:
import os

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/translate'
DRIVE_OUTPUT_DIR  = '/content/drive/MyDrive/translate_outputs'

WORK_DIR = '/content/translate'

MODEL_NAME     = 'Qwen/Qwen3-1.7B'
TRAIN_JSONL    = '/content/translate/data/train.jsonl'
VALID_JSONL    = '/content/translate/data/tst2012.jsonl'
TEST_JSONL     = '/content/translate/data/tst2013.jsonl'

SFT_OUTPUT_DIR   = os.path.join(DRIVE_OUTPUT_DIR, 'Qwen-1.7B-SFT-VI2EN')
GRPO_OUTPUT_DIR  = os.path.join(DRIVE_OUTPUT_DIR, 'Qwen-1.7B-GRPO-VI2EN')
RESULTS_FILE     = os.path.join(DRIVE_OUTPUT_DIR, 'results.jsonl')
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
assert os.path.isdir(DRIVE_PROJECT_DIR), f'Project dir not found in Drive: {DRIVE_PROJECT_DIR}'
print('Project:', DRIVE_PROJECT_DIR)
print('Output (Drive):', DRIVE_OUTPUT_DIR)

Project: /content/drive/MyDrive/translate
Output (Drive): /content/drive/MyDrive/translate_outputs


## 4. Install dependencies


In [ ]:
!pip install -q -U transformers==4.46.3 trl==0.12.1 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 sacrebleu==2.4.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 123.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beh

## 5. Sanity-check GPU and data

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
!head -1 /content/translate/data/train.jsonl
!wc -l /content/translate/data/train.jsonl /content/translate/data/tst2012.jsonl /content/translate/data/tst2013.jsonl

CUDA available: True
Device: NVIDIA A100-SXM4-40GB
bf16 supported: True
{"input": "đằng_sau :arg khoa_học :arg ( tiêu_đề :quant num_ :topic khí_hậu )", "vi": "Khoa_học đằng sau một tiêu_đề về khí_hậu", "output": "Rachel Pike : The science behind a climate headline"}
  131261 /content/translate/data/train.jsonl
    1528 /content/translate/data/tst2012.jsonl
    1253 /content/translate/data/tst2013.jsonl
  134042 total


## 6. Preview the prompt that the model will see

This double-checks the dataset loader is picking `input` (AMR), `vi`, and `output` correctly.

In [ ]:
from translate.viamr.dataset import get_data

ds = get_data(TRAIN_JSONL, type='sft')
print('Examples:', len(ds))
ex = ds[0]
print('--- system ---')
print(ex['prompt'][0]['content'])
print('--- user ---')
print(ex['prompt'][1]['content'])
print('--- target ---')
print(ex['completion'][0]['content'])

Loaded 131261 examples. Max input words: 386, max output words: 100
Examples: 131261
--- system ---
You are a professional Vietnamese-to-English translator with expertise in Abstract Meaning Representation (AMR). For each example you will receive two inputs:
1. An AMR expression describing the semantic structure of the sentence (predicates, arguments, roles such as :arg, :mod, :time, :location, ...).
2. The original Vietnamese sentence.

Use the AMR as a semantic scaffold to disambiguate word senses, recover implicit arguments, and preserve relations (who did what to whom, when, where, why). The Vietnamese sentence is the surface form you are translating; the AMR must not introduce content that is not supported by the sentence.

Produce a fluent, faithful English translation. Output only the English translation — no AMR, no Vietnamese, no explanations, no tags.
--- user ---
Translate the following Vietnamese sentence to English, using the AMR as semantic guidance.
AMR: đằng_sau :arg kh

## 7. SFT fine-tuning

We call the training entrypoint as a Python module so the same code path runs here and in `scripts/train_sft.sh`. Tune `PER_DEVICE_BS`, `GRAD_ACC`, and `EPOCHS` to fit your GPU.

In [ ]:
!pip install --upgrade transformers accelerate

In [ ]:
!pip install --upgrade trl transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 173.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
#!/bin/bash
!set -e

# Run from the repo root so `viamr` is importable.
!cd "$(dirname "$0")/.."

!echo "Running SFT training (vi -> en)..."
!export CUDA_VISIBLE_DEVICES=0,1

!python -m translate.viamr.training.sft \
    --dataset1_path "translate/data/train.jsonl" \
    --output_dir "/content/drive/MyDrive/translate_outputs/Qwen-1.7B-SFT-VI2EN" \
    --model_name "/content/drive/MyDrive/translate_outputs/Qwen-1.7B-SFT-VI2EN/checkpoint-2500" \
    --learning_rate 2e-5 \
    --adam_beta1 0.9 \
    --adam_beta2 0.999 \
    --weight_decay 0.01 \
    --warmup_steps 500 \
    --lr_scheduler_type cosine \
    --logging_steps 10 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --max_input_length 512 \
    --num_train_epochs 2 \
    --save_steps 500 \
    --eval_steps 500 \
    --lora_r 32 \
    --lora_alpha 64 \
    --lora_dropout 0.1 \
    --use_lora 0 \
    2>&1 | tee /content/drive/MyDrive/translate_outputs/Qwen-1.7B-SFT-VI2EN/Qwen-SFT-VI2EN.log


Running SFT training (vi -> en)...
`torch_dtype` is deprecated! Use `dtype` instead!
Loaded 131261 examples. Max input words: 386, max output words: 100
Tokenizing train dataset: 100%|██████████| 131261/131261 [04:58<00:00, 440.38 examples/s]
{'loss': '0.9651', 'grad_norm': '6.531', 'learning_rate': '3.6e-07', 'entropy': '2.648', 'num_tokens': '5.012e+04', 'mean_token_accuracy': '0.7757', 'epoch': '0.001219'}
{'loss': '0.9679', 'grad_norm': '10.31', 'learning_rate': '7.6e-07', 'entropy': '2.646', 'num_tokens': '1.008e+05', 'mean_token_accuracy': '0.7695', 'epoch': '0.002438'}
{'loss': '0.9184', 'grad_norm': '7.312', 'learning_rate': '1.16e-06', 'entropy': '2.64', 'num_tokens': '1.515e+05', 'mean_token_accuracy': '0.785', 'epoch': '0.003657'}
{'loss': '0.8985', 'grad_norm': '7.344', 'learning_rate': '1.56e-06', 'entropy': '2.645', 'num_tokens': '2.024e+05', 'mean_token_accuracy': '0.7842', 'epoch': '0.004876'}
{'loss': '0.9226', 'grad_norm': '7.406', 'learning_rate': '1.96e-06', 'entrop

## 8. Inference on the test set


In [ ]:
!python -m translate.viamr.inference \
  --input_file "$TEST_JSONL" \
  --output_file "$RESULTS_FILE" \
  --model_name "$SFT_OUTPUT_DIR" \
  --max_new_tokens 512

Streaming output truncated to the last 5000 lines.

</think>

I realized that the man I loved would kill me if I left him .
[254] Vì vậy tôi phát vỡ sự yên lặng . -> <think>

</think>

So I shattered the silence .
[255] Tôi kể với mọi người : cánh sát , những người láng giềng , bạn bè và gia đình tôi , những người hoàn toàn xa lạ , và tôi đứng đây hôm nay bởi vì bạn đều đang giúp tôi . -> <think>

</think>

I tell you , my neighbors , my friends and my family , who are completely strangers to me , and I stand here today because you &apos;re helping me .
[256] Chúng ta có quan niệm rập khuôn về nạn nhân như những tiêu đề đáng sợ , những phụ nữ tự huỷ hoại mình , những điều tốt bị làm tổn thương . -> <think>

</think>

We have stereotypes of victims as scary headlines , as self-destructive women , as things that are damaged .
[257] Câu hỏi , & quot ; Tai sao cô ấy ở lại ? & quot ; -> <think>

</think>

The question , &quot; Why did she stay ? &quot;
[258] là lí lẽ của một số người , & qu

In [ ]:
!echo "Moving outputs to Google Drive..."

!cp -r /content/outputs/Qwen-1.7B-SFT-VI2EN /content/drive/MyDrive/translate_outputs/

!echo "Done!"

## 9. Corpus BLEU

In [8]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.0 MB/s eta 0:00:00


In [7]:
import json
import re
import html

INPUT_FILE = "/content/drive/MyDrive/translate_outputs/results.jsonl"
OUTPUT_FIX_FILE = "/content/drive/MyDrive/translate_outputs/results_clean.jsonl"

def clean_pred(text):
    if text is None:
        return ""
    text = html.unescape(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"\s+", " ", text).strip()
    return text

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FIX_FILE, "w", encoding="utf-8") as fout:

    for line in fin:
        item = json.loads(line)

        if "pred" in item:
            item["pred"] = clean_pred(item["pred"])

        fout.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Done! Clean file saved to:", OUTPUT_FIX_FILE)

Done! Clean file saved to: /content/drive/MyDrive/translate_outputs/results_clean.jsonl


In [9]:
!python -m translate.viamr.scoring \
  --predict_file "$OUTPUT_FIX_FILE"

Number of predictions: 1253, Number of references: 1253
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
Corpus BLEU: 33.4138
BLEU = 33.41 64.4/40.8/27.9/19.6 (BP = 0.965 ratio = 0.965 hyp_len = 26554 ref_len = 27509)


## 10. (Optional) GRPO stage on top of SFT


In [14]:
!WANDB_MODE=disabled python -m translate.viamr.training.grpo \
  --dataset1_path "$TRAIN_JSONL" \
  --output_dir "$GRPO_OUTPUT_DIR" \
  --model_name "$SFT_OUTPUT_DIR" \
  \
  --learning_rate 5e-6 \
  --warmup_steps 200 \
  --lr_scheduler_type cosine \
  \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 16 \
  \
  --num_generations 4 \
  \
  --max_prompt_length 384 \
  --max_completion_length 512 \
  \
  --num_train_epochs 2 \
  --save_steps 500 \
  --logging_steps 1 \
  \
  --use_lora 1 \
  --lora_r 32 \
  --lora_alpha 64 \
  --lora_dropout 0.05 \
  \
  --wandb_project vi2en-translation \
  --wandb_run_name grpo-bleu-final-v2

Loaded 131261 examples. Max input words: 386, max output words: 100
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 310/310 [00:00<00:00, 2718.46it/s]
  0% 0/32814 [00:00<?, ?it/s]BLEU: 0.1384 | pred: Air pollution per school , global warming -- all of these things , the consequen | gold: Pollution , global warming , these things -- the consequences are distant in tim
BLEU: 0.1350 | pred: Pollution in each field of study , global warming , all these things , the conse | gold: Pollution , global warming , these things -- the consequences are distant in tim
BLEU: 0.0687 | pred: Pollution , climate change , all of these consequences , far in space and time . | gold: Pollution , global warming , these things -- the consequences are distant in tim
BLEU: 0.1352 | pred: Polluted schools , global warming , all these things , and their consequences ar | gold: Pollution , global warming , these things -- the consequences are distant in tim
BLEU: 0.1277 | pred: And I want 